# Klasifikasi Gambar dengan CNN dan Tensorflow/Keras

Notebook ini adalah versi bahasa Indonesia dari tutorial TensorFlow **Image Classification**.  
Fokus tutorial ini adalah membangun model sederhana untuk mengenali gambar bunga menggunakan **Convolutional Neural Network (CNN)**. Tutorial aslinya dapat diakses [di sini](https://www.tensorflow.org/tutorials/images/classification).

Dataset yang digunakan adalah `flower_photos`, yaitu kumpulan gambar bunga dengan 5 kelas:

1. `daisy`
2. `dandelion`
3. `roses`
4. `sunflowers`
5. `tulips`

## Apa yang akan dipelajari?

Setelah menyelesaikan notebook ini, Anda diharapkan dapat:

- memahami struktur dataset gambar berbasis folder,
- memuat dataset gambar menggunakan `image_dataset_from_directory`,
- melihat bentuk tensor gambar,
- melakukan normalisasi piksel gambar,
- membangun model CNN sederhana,
- membaca grafik training dan validation,
- mengenali gejala **overfitting**,
- mengurangi overfitting dengan **data augmentation** dan **dropout**,
- melakukan prediksi terhadap gambar baru,
- menyimpan model dan mengonversinya ke TensorFlow Lite secara opsional.

> Catatan pemula: Jangan khawatir jika istilah CNN, epoch, tensor, atau overfitting terasa baru. Notebook ini akan menjelaskan konsepnya secara bertahap.

## 1. Import Library

Pertama, kita memanggil beberapa library yang dibutuhkan.

- `tensorflow`: framework utama untuk deep learning.
- `keras`: API tingkat tinggi di dalam TensorFlow untuk membuat model neural network.
- `matplotlib`: untuk menampilkan gambar dan grafik.
- `numpy`: untuk operasi numerik.
- `pathlib`: untuk mengelola path/folder dataset.
- `PIL`: untuk membuka file gambar.

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import PIL
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

print("Versi TensorFlow:", tf.__version__)

## 2. Mengunduh dan Mengeksplorasi Dataset

Dataset gambar bunga akan diunduh dari server TensorFlow. Dataset ini sudah disusun dalam folder berdasarkan kelasnya.

Strukturnya kurang lebih seperti ini:

```text
flower_photos/
    daisy/
    dandelion/
    roses/
    sunflowers/
    tulips/
```

Dalam klasifikasi gambar menggunakan folder, nama folder biasanya dianggap sebagai nama kelas.  
Misalnya, semua gambar di folder `roses` akan diberi label `roses`.

In [ ]:
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

data_dir = tf.keras.utils.get_file(
    "flower_photos.tar",
    origin=dataset_url,
    extract=True
)

data_dir = pathlib.Path(data_dir).with_suffix("")
print("Lokasi dataset:", data_dir)

Sekarang kita hitung jumlah total gambar di dalam dataset.

Kode `data_dir.glob('*/*.jpg')` berarti:

- `*` pertama: masuk ke semua folder kelas,
- `*.jpg`: ambil semua file gambar berformat `.jpg`.

In [ ]:
image_count = len(list(data_dir.glob("*/*.jpg")))
print("Jumlah total gambar:", image_count)

## 3. Melihat Contoh Gambar

Sebelum membuat model, biasakan untuk **melihat data** terlebih dahulu.

Ini penting karena dalam machine learning kita tidak boleh langsung melatih model tanpa memahami data.  
Kita perlu memastikan bahwa gambar memang benar, kelasnya masuk akal, dan tidak ada struktur folder yang salah.

In [ ]:
roses = list(data_dir.glob("roses/*"))
tulips = list(data_dir.glob("tulips/*"))

print("Contoh file roses:", roses[0])
print("Contoh file tulips:", tulips[0])

PIL.Image.open(str(roses[0]))

In [ ]:
PIL.Image.open(str(tulips[0]))

## 4. Memuat Dataset dengan `image_dataset_from_directory`

TensorFlow menyediakan fungsi praktis bernama:

```python
tf.keras.utils.image_dataset_from_directory
```

Fungsi ini membaca gambar dari folder, mengubah gambar menjadi tensor, dan membuat label berdasarkan nama folder.

Kita akan menggunakan:

- `image_size=(180, 180)`: semua gambar diubah menjadi ukuran 180 x 180 piksel,
- `batch_size=32`: data diproses per kelompok berisi 32 gambar,
- `validation_split=0.2`: 20% data digunakan untuk validasi,
- `subset="training"` dan `subset="validation"`: membagi dataset menjadi training dan validation,
- `seed=123`: agar pembagian data dapat direproduksi.

In [ ]:
batch_size = 32
img_height = 180
img_width = 180
seed = 123

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

Nama kelas dapat diambil dari atribut `class_names`.

Perhatikan bahwa urutan kelas biasanya mengikuti urutan alfabet berdasarkan nama folder.

In [ ]:
class_names = train_ds.class_names
num_classes = len(class_names)

print("Daftar kelas:", class_names)
print("Jumlah kelas:", num_classes)

## 5. Visualisasi Batch Data

Satu batch berisi beberapa gambar dan label.  
Mari kita tampilkan 9 gambar pertama dari dataset training.

In [ ]:
plt.figure(figsize=(10, 10))

for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

plt.show()

## 6. Memahami Bentuk Tensor Gambar

Dalam deep learning, gambar direpresentasikan sebagai tensor.

Untuk gambar berwarna RGB, bentuk satu gambar adalah:

```text
tinggi x lebar x channel
```

Karena kita menggunakan ukuran 180 x 180 dan gambar RGB memiliki 3 channel, maka bentuk satu gambar adalah:

```text
180 x 180 x 3
```

Namun data diproses dalam batch. Jika batch size adalah 32, bentuk batch gambar adalah:

```text
32 x 180 x 180 x 3
```

In [ ]:
for image_batch, labels_batch in train_ds.take(1):
    print("Shape batch gambar:", image_batch.shape)
    print("Shape batch label:", labels_batch.shape)
    print("Contoh label pertama:", labels_batch[0].numpy())
    break

## 7. Optimasi Pipeline Data

Saat training, model membutuhkan data secara terus-menerus. Jika pembacaan data lambat, GPU/CPU bisa menunggu data terlalu lama.

Kita gunakan:

- `cache()`: menyimpan data setelah pertama kali dibaca,
- `prefetch()`: menyiapkan batch berikutnya saat model sedang memproses batch saat ini,
- `AUTOTUNE`: TensorFlow memilih pengaturan yang efisien secara otomatis.

Untuk dataset kecil seperti `flower_photos`, `cache()` cukup aman.  
Untuk dataset sangat besar, cache di memori perlu dipertimbangkan ulang.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 8. Normalisasi Nilai Piksel

Nilai piksel gambar RGB biasanya berada pada rentang:

```text
0 sampai 255
```

Neural network biasanya lebih mudah dilatih jika input berada pada rentang kecil, misalnya:

```text
0 sampai 1
```

Kita dapat melakukan normalisasi dengan:

```python
layers.Rescaling(1./255)
```

Dalam notebook ini, normalisasi akan dimasukkan langsung ke dalam model agar proses deployment lebih sederhana.

In [ ]:
normalization_layer = layers.Rescaling(1./255)

# Contoh pengecekan normalisasi pada satu batch
for images, labels in train_ds.take(1):
    normalized_images = normalization_layer(images)
    print("Nilai piksel sebelum normalisasi:", np.min(images.numpy()), "sampai", np.max(images.numpy()))
    print("Nilai piksel setelah normalisasi:", np.min(normalized_images.numpy()), "sampai", np.max(normalized_images.numpy()))
    break

## 9. Membangun Model CNN Dasar

Sekarang kita membuat model CNN sederhana.

Model ini terdiri dari:

1. `Rescaling`: normalisasi piksel dari 0-255 menjadi 0-1.
2. `Conv2D`: mencari pola visual seperti garis, warna, tekstur, atau bentuk.
3. `MaxPooling2D`: mengecilkan ukuran feature map agar komputasi lebih ringan.
4. `Flatten`: mengubah hasil 2D/3D menjadi vektor 1D.
5. `Dense`: melakukan klasifikasi berdasarkan fitur yang ditemukan.
6. Output layer dengan jumlah neuron sama dengan jumlah kelas.

Output terakhir tidak memakai `softmax` karena kita akan menggunakan:

```python
SparseCategoricalCrossentropy(from_logits=True)
```

Artinya, model menghasilkan **logit**, yaitu skor mentah sebelum diubah menjadi probabilitas.

In [ ]:
model_baseline = keras.Sequential([
    keras.Input(shape=(img_height, img_width, 3)),
    layers.Rescaling(1./255),

    layers.Conv2D(16, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes)
])

model_baseline.summary()

## 10. Compile Model

Sebelum training, model perlu dikompilasi.

Kita menentukan:

- `optimizer="adam"`: algoritma untuk memperbarui bobot model.
- `loss=SparseCategoricalCrossentropy(from_logits=True)`: fungsi loss untuk klasifikasi multi-kelas dengan label integer.
- `metrics=["accuracy"]`: metrik untuk memantau akurasi.

In [ ]:
model_baseline.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

## 11. Training Model Dasar

`epoch` adalah satu putaran penuh model melihat seluruh data training.

Untuk pemula, kita gunakan jumlah epoch yang tidak terlalu besar agar training tidak terlalu lama.  
Jika komputer Anda cukup kuat, Anda dapat meningkatkan `epochs_baseline` menjadi 10.

In [ ]:
epochs_baseline = 5

history_baseline = model_baseline.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs_baseline
)

## 12. Visualisasi Hasil Training

Kita akan membuat grafik:

- training accuracy,
- validation accuracy,
- training loss,
- validation loss.

Cara membaca grafik:

- Jika training accuracy naik tetapi validation accuracy stagnan atau turun, kemungkinan terjadi **overfitting**.
- Jika training dan validation sama-sama rendah, kemungkinan model **underfitting**.
- Jika training dan validation meningkat bersama, model belajar dengan baik.

In [ ]:
def plot_training_history(history, title="Hasil Training"):
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]

    loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    epochs_range = range(len(acc))

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="Training Accuracy")
    plt.plot(epochs_range, val_acc, label="Validation Accuracy")
    plt.legend(loc="lower right")
    plt.title(f"{title}: Accuracy")

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="Training Loss")
    plt.plot(epochs_range, val_loss, label="Validation Loss")
    plt.legend(loc="upper right")
    plt.title(f"{title}: Loss")

    plt.show()

plot_training_history(history_baseline, title="Model Dasar")

## 13. Overfitting: Apa yang Terjadi?

**Overfitting** terjadi ketika model terlalu hafal data training, tetapi kurang baik saat menghadapi data baru.

Contoh sederhana:

- Mahasiswa menghafal jawaban latihan, tetapi tidak memahami konsep.
- Saat soal ujian sedikit berbeda, jawabannya salah.

Dalam konteks CNN:

- Model mungkin menghafal detail kecil dari gambar training,
- tetapi gagal mengenali gambar bunga baru yang pencahayaan, posisi, atau ukurannya berbeda.

Dua teknik yang umum digunakan untuk mengurangi overfitting:

1. **Data augmentation**
2. **Dropout**

## 14. Data Augmentation

**Data augmentation** membuat variasi gambar baru dari gambar yang sudah ada.

Contoh transformasi:

- gambar dibalik secara horizontal,
- gambar diputar sedikit,
- gambar diperbesar atau diperkecil sedikit.

Tujuannya bukan membuat kelas baru, melainkan membuat model melihat variasi yang lebih beragam sehingga lebih tahan terhadap perbedaan gambar baru.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

Mari kita lihat efek data augmentation pada satu gambar yang sama.

Gambar asli akan diubah beberapa kali secara acak.

In [ ]:
plt.figure(figsize=(10, 10))

for images, labels in train_ds.take(1):
    first_image = images[0]

    for i in range(9):
        augmented_image = data_augmentation(tf.expand_dims(first_image, 0), training=True)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented_image[0].numpy().astype("uint8"))
        plt.axis("off")

plt.show()

## 15. Dropout

**Dropout** adalah teknik regularisasi yang mematikan sebagian unit/neuron secara acak saat training.

Misalnya:

```python
layers.Dropout(0.2)
```

Artinya, sekitar 20% unit akan dimatikan secara acak selama training.

Tujuannya adalah mencegah model terlalu bergantung pada neuron tertentu sehingga model belajar pola yang lebih umum.

## 16. Membangun Model yang Ditingkatkan

Sekarang kita gabungkan:

- data augmentation,
- rescaling,
- CNN,
- dropout.

Model ini diharapkan lebih baik dalam mengurangi overfitting dibanding model dasar.

In [ ]:
model_improved = keras.Sequential([
    keras.Input(shape=(img_height, img_width, 3)),
    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(16, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Dropout(0.2),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, name="outputs")
])

model_improved.summary()

In [ ]:
model_improved.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

Training model yang ditingkatkan.

Anda dapat menaikkan `epochs_improved` menjadi 10 atau 15 jika ingin hasil yang lebih stabil.

In [ ]:
epochs_improved = 5

history_improved = model_improved.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs_improved
)

In [ ]:
plot_training_history(history_improved, title="Model dengan Augmentation dan Dropout")

## 17. Membandingkan Model Dasar dan Model yang Ditingkatkan

Bagian ini membandingkan akurasi validasi terakhir dari kedua model.

Perlu diperhatikan:

- Hasil dapat berbeda pada setiap mesin.
- Jika epoch terlalu sedikit, model improved belum tentu langsung terlihat lebih baik.
- Yang penting adalah memahami pola: data augmentation dan dropout biasanya membantu mengurangi overfitting.

In [ ]:
baseline_val_acc = history_baseline.history["val_accuracy"][-1]
improved_val_acc = history_improved.history["val_accuracy"][-1]

print(f"Validation accuracy model dasar      : {baseline_val_acc:.4f}")
print(f"Validation accuracy model improved   : {improved_val_acc:.4f}")

## 18. Prediksi Gambar Baru

Sekarang kita gunakan model untuk mengklasifikasikan gambar baru yang tidak termasuk dalam dataset training/validation.

Langkahnya:

1. Unduh gambar baru.
2. Ubah ukuran gambar menjadi 180 x 180.
3. Ubah gambar menjadi array.
4. Tambahkan dimensi batch.
5. Prediksi dengan model.
6. Ubah logit menjadi probabilitas menggunakan `softmax`.

In [ ]:
sunflower_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/592px-Red_sunflower.jpg"

sunflower_path = tf.keras.utils.get_file(
    "Red_sunflower",
    origin=sunflower_url
)

img = tf.keras.utils.load_img(
    sunflower_path,
    target_size=(img_height, img_width)
)

plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
img_array = tf.keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)  # membuat batch berisi 1 gambar

predictions = model_improved.predict(img_array)
score = tf.nn.softmax(predictions[0])

predicted_class = class_names[np.argmax(score)]
confidence = 100 * np.max(score)

print(f"Gambar ini paling mungkin termasuk kelas: {predicted_class}")
print(f"Tingkat keyakinan model: {confidence:.2f}%")

## 19. Melihat Probabilitas Semua Kelas

Daripada hanya melihat kelas dengan probabilitas tertinggi, kita juga dapat melihat skor untuk semua kelas.

In [ ]:
for class_name, probability in zip(class_names, score.numpy()):
    print(f"{class_name:12s}: {probability:.4f}")

## 20. Membuat Fungsi Prediksi Sendiri

Agar lebih praktis, kita buat fungsi untuk memprediksi satu gambar dari path file.

Fungsi ini berguna jika Anda ingin mencoba gambar lain.

In [ ]:
def predict_image(image_path, model, class_names, img_height=180, img_width=180):
    img = tf.keras.utils.load_img(
        image_path,
        target_size=(img_height, img_width)
    )

    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array)
    score = tf.nn.softmax(predictions[0])

    predicted_class = class_names[np.argmax(score)]
    confidence = 100 * np.max(score)

    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Prediksi: {predicted_class} ({confidence:.2f}%)")
    plt.show()

    return predicted_class, confidence

predict_image(sunflower_path, model_improved, class_names, img_height, img_width)

## 21. Menyimpan Model

Setelah model dilatih, kita dapat menyimpannya agar tidak perlu training ulang.

Format `.keras` adalah format penyimpanan model Keras modern.

In [ ]:
model_path = "model_klasifikasi_bunga.keras"
model_improved.save(model_path)

print("Model berhasil disimpan ke:", model_path)

Untuk memuat ulang model:

In [ ]:
loaded_model = keras.models.load_model(model_path)

# Tes singkat model yang dimuat ulang
predictions_loaded = loaded_model.predict(img_array)
score_loaded = tf.nn.softmax(predictions_loaded[0])

print("Prediksi dari model yang dimuat ulang:", class_names[np.argmax(score_loaded)])

## 22. Opsional: Konversi ke TensorFlow Lite

TensorFlow Lite digunakan untuk menjalankan model di perangkat seperti:

- Android,
- iOS,
- perangkat IoT,
- embedded system.

Bagian ini opsional. Untuk pemula, cukup pahami bahwa model Keras dapat dikonversi menjadi file `.tflite`.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_improved)
tflite_model = converter.convert()

tflite_model_path = "model_klasifikasi_bunga.tflite"

with open(tflite_model_path, "wb") as f:
    f.write(tflite_model)

print("Model TensorFlow Lite disimpan ke:", tflite_model_path)

## 23. Ringkasan Konsep Penting

Berikut ringkasan materi yang sudah dipelajari:

| Konsep | Penjelasan Sederhana |
|---|---|
| Dataset gambar | Kumpulan gambar yang memiliki label kelas |
| Folder kelas | Nama folder digunakan sebagai label |
| Tensor gambar | Representasi gambar dalam bentuk angka |
| RGB | Tiga channel warna: red, green, blue |
| CNN | Neural network yang cocok untuk memproses gambar |
| Conv2D | Layer untuk mencari pola visual |
| MaxPooling2D | Layer untuk meringkas feature map |
| Rescaling | Mengubah piksel 0-255 menjadi 0-1 |
| Overfitting | Model terlalu hafal data training |
| Data augmentation | Membuat variasi gambar training secara acak |
| Dropout | Mematikan sebagian neuron saat training |
| Softmax | Mengubah skor model menjadi probabilitas |
| TensorFlow Lite | Format model untuk perangkat ringan/mobile |

## Latihan 1. Ubah Jumlah Epoch

Ubah:

```python
epochs_baseline = 5
epochs_improved = 5
```

menjadi 10 atau 15.

Pertanyaan:

1. Apakah training accuracy meningkat?
2. Apakah validation accuracy ikut meningkat?
3. Apakah terlihat tanda overfitting?

---

## Latihan 2. Ubah Ukuran Gambar

Ubah:

```python
img_height = 180
img_width = 180
```

menjadi:

```python
img_height = 128
img_width = 128
```

atau:

```python
img_height = 224
img_width = 224
```

Pertanyaan:

1. Apakah training menjadi lebih cepat atau lebih lambat?
2. Apakah akurasi berubah?
3. Apa hubungan ukuran gambar dengan biaya komputasi?

---

## Latihan 3. Tambahkan Data Augmentation

Tambahkan layer berikut ke `data_augmentation`:

```python
layers.RandomContrast(0.2)
```

Pertanyaan:

1. Apakah model menjadi lebih baik?
2. Apakah augmentation terlalu kuat dapat merusak gambar?

---

## Latihan 4. Ubah Dropout

Ubah:

```python
layers.Dropout(0.2)
```

menjadi:

```python
layers.Dropout(0.4)
```

Pertanyaan:

1. Apakah overfitting berkurang?
2. Apakah training accuracy turun terlalu jauh?
3. Kapan dropout terlalu besar menjadi masalah?

---

## Latihan 5. Prediksi Gambar Sendiri

Coba gunakan gambar bunga dari internet atau file lokal Anda sendiri.

Gunakan fungsi:

```python
predict_image("path_gambar_anda.jpg", model_improved, class_names)
```

Pertanyaan:

1. Apakah prediksi benar?
2. Berapa confidence model?
3. Apakah confidence tinggi selalu berarti prediksi benar?

---

## Latihan 6. Analisis Kesalahan Model

Ambil beberapa gambar validation yang salah diprediksi.

Petunjuk:

1. Prediksi batch gambar dari `val_ds`.
2. Bandingkan `np.argmax(prediction)` dengan label asli.
3. Tampilkan gambar yang salah.

Pertanyaan:

1. Kelas apa yang paling sering tertukar?
2. Apakah kesalahan terjadi karena bentuk bunga mirip?
3. Apakah background gambar memengaruhi prediksi?

---

## Latihan 7. Gunakan Dataset Sendiri

Buat folder seperti ini:

```text
my_dataset/
    class_1/
    class_2/
    class_3/
```

Lalu ganti:

```python
data_dir = pathlib.Path("my_dataset")
```

Pertanyaan:

1. Apakah jumlah gambar per kelas seimbang?
2. Apakah semua gambar memiliki kualitas yang cukup?
3. Apakah nama folder sudah mewakili label yang benar?

# Penutup

Notebook ini menunjukkan workflow dasar klasifikasi gambar:

```text
Unduh data
→ eksplorasi data
→ buat dataset training/validation
→ normalisasi gambar
→ bangun CNN
→ training
→ evaluasi
→ perbaiki overfitting
→ prediksi gambar baru
→ simpan model
```

Dalam praktik nyata, peningkatan performa biasanya dilakukan dengan:

- menambah data,
- membersihkan label yang salah,
- menggunakan arsitektur CNN yang lebih kuat,
- menerapkan transfer learning,
- melakukan tuning hyperparameter,
- mengevaluasi model dengan data test yang benar-benar terpisah.